# exp-04 — panel-major or check-major, on the cleaned contracts

Reads the runs under `experiments/runs/exp-04-panel-major/`, scores every leaf
that is not scored yet, and compares, within each arrangement of the skill DAG,
the **panel-major** (PM) answer against the **check-major** (CM) one.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-04-panel-major/run.py
```

which is 1,140 sessions: two entry skills × three arrangements × 38 figures ×
5 replicates.

The claims this is evidence for, the margins and the rule that turns the
results into a choice of contract are fixed in the preregistration,
[`thinking/experiments/exp-04-panel-major.md`](../../thinking/experiments/exp-04-panel-major.md).
This notebook computes the numbers those rules read; it does not decide.

## The six conditions

| condition | entry skill | arrangement | arm directory |
|---|---|---|---|
| `CM A\|B\|C_i <- D` | `do-fig-checklist-cm` | checks at v1, A and B inline | `cm/pinned` |
| `CM A\|B <- C_i <- D` | `do-fig-checklist-cm` | checks at v3, calling `classify-panels` v1 | `cm/<checks>@v3` |
| `CM A <- B <- C_i <- D` | `do-fig-checklist-cm` | checks at v3, `classify-panels` v2 calling `identify-panels` | `cm/classify-panels@v2-<checks>@v3` |
| `PM …` | `do-fig-checklist-pm` | the same three | `pm/…` |

The two entry skills' prose is identical but for the name; each owns one
contract, derived from the three checks' cleaned contracts and verified to
score every check exactly as its own contract does
(`experiments/exp-04-panel-major/build_contract.py`). Both shapes are rewritten
below into per-check terms — `check`, `outputs[].x`, `outputs` — so one set of
code reads both. In PM, the shared `panel_label` and the one row set are
**counted towards every check**, as the verification does.

## The gold this experiment is scored against

**`gold-v4`**: `gold-v3`, the gold at the close of the contract cleanup, with
the merged gold of the two entry skills added. This cell pins the scorer to it,
read from a gold-only snapshot (`soda_mmqc/gold.py`). It must run before
anything else imports `soda_mmqc`.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v4"
pin_gold(GOLD_TAG)

In [ ]:
import importlib.util
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.config import gold_root
from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import (
    arm_contrast,
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_stacked_counts,
    rate_contrast,
    scores_frame,
)
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)

EXPERIMENT = Path("../../experiments/exp-04-panel-major")


def _load(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


#: The runner names the arrangements, the entry skills and the gold tag;
#: panels.py counts spurious panels. Imported, not restated.
exp04 = _load("exp04_run", EXPERIMENT / "run.py")
panels = _load("exp04_panels", EXPERIMENT / "panels.py")
assert exp04.GOLD_TAG == GOLD_TAG, "the runner and this notebook name different gold"

CHECKLIST, ENTRIES, CHECKS = exp04.CHECKLIST, exp04.ENTRIES, exp04.CHECKS
CLASSIFY, IDENTIFY = exp04.CLASSIFY, exp04.IDENTIFY
ARRANGEMENTS = tuple(exp04.ARRANGEMENTS)
SHAPES = ("cm", "pm")

#: The full run. Set EXP04_ROOT to `.../smoke` to exercise the notebook on the
#: smoke test.
RUNS = Path("../../experiments/runs/exp-04-panel-major")
ROOT = Path(os.environ.get("EXP04_ROOT", RUNS))

#: Where gold is read: the pinned snapshot, not the live tree.
GOLD = gold_root()

#: Margins, fixed in the preregistration.
DELTA = {"layer S": 0.0125, "layer 1": 0.02, "layer 2": 0.02}

#: Pre-declared as expected to be uninformative: reported, not counted, if
#: their interval is wider than the margin.
UNINFORMATIVE = {
    ("layer S", "micrograph-scale-bar", None, "A|B <- C_i <- D"),
    ("layer S", "micrograph-scale-bar", None, "A <- B <- C_i <- D"),
    *(("layer 2", "micrograph-scale-bar", "outputs[].from_the_image", a) for a in ARRANGEMENTS),
}

#: Seeded so the intervals are reproducible.
SEED = 20261005
RESAMPLES = 10_000

LABEL_TO_ARRANGEMENT = {exp04.arm_label(unpin): name for name, unpin in exp04.ARRANGEMENTS.items()}


def condition(shape: str, arrangement: str) -> str:
    return f"{shape.upper()} {arrangement}"


CONDITION_ORDER = tuple(condition(s, a) for a in ARRANGEMENTS for s in SHAPES)
#: Within each arrangement, PM (variant) against CM (baseline).
PAIRS = {condition("pm", a): condition("cm", a) for a in ARRANGEMENTS}


def _cm_owner(value: str) -> str | None:
    for check in CHECKS:
        if value == check or value.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def to_per_check(frame: pd.DataFrame, shape: str) -> pd.DataFrame:
    """Rewrite one shape's frame into per-check terms, and label its condition.

    CM: `<check>[].x` -> (`check`, `outputs[].x`); list key `<check>` -> `outputs`.
    PM: `outputs[].<check>.x` -> (`check`, `outputs[].x`); the shared
    `outputs[].panel_label` and the one list key `outputs` are copied to every
    check, which is how a PM answer reads per check.
    """
    frame = frame.copy()
    frame["arrangement"] = [LABEL_TO_ARRANGEMENT[a] for a in frame["arm"]]
    frame["condition"] = [condition(shape, a) for a in frame["arrangement"]]
    key = "property" if "property" in frame else "list_key"
    if shape == "cm":
        frame["check"] = [_cm_owner(v) for v in frame[key]]
        if frame["check"].isna().any():
            raise ValueError(f"CM {key}s owned by no check: "
                             f"{sorted(frame.loc[frame['check'].isna(), key].unique())}")
        for column in ("property", "list_key"):
            if column in frame:
                frame[column] = ["outputs" + v[len(c):] for v, c in zip(frame[column], frame["check"])]
        return frame
    parts = []
    for check in CHECKS:
        if key == "list_key":
            part = frame[frame["list_key"] == "outputs"].copy()
        else:
            own = frame[key].str.startswith(f"outputs[].{check}.")
            shared = frame[key] == "outputs[].panel_label"
            part = frame[own | shared].copy()
            part[key] = part[key].str.replace(f"outputs[].{check}.", "outputs[].", regex=False)
        part["check"] = check
        parts.append(part)
    out = pd.concat(parts, ignore_index=True)
    if key == "property":
        owned = {p for p in frame[key]
                 if p == "outputs[].panel_label" or any(p.startswith(f"outputs[].{c}.") for c in CHECKS)}
        if set(frame[key]) - owned:
            raise ValueError(f"PM properties owned by no check: {sorted(set(frame[key]) - owned)}")
    return out


def run_roots():
    """(shape, entry skill, root) for both shapes."""
    for shape in SHAPES:
        yield shape, ENTRIES[shape], ROOT / shape


print(ROOT.resolve())

## Score every leaf that is not scored yet

Idempotent, and it refuses to trust an `analysis.json` older than the
predictions under it, or older than the gold it was scored against.

In [ ]:
stale = []
for shape, entry, root in run_roots():
    if not root.is_dir():
        print(f"no runs yet under {root}")
        continue
    for arm, replicate, leaf in iter_leaves(root):
        analysis = leaf / ANALYSIS_FILENAME
        if not analysis.is_file():
            score_check(CHECKLIST, entry, leaf, save=True)
            print(f"scored {shape} / {LABEL_TO_ARRANGEMENT[arm]} / rep-{replicate:02d}")
            continue
        predictions = list(leaf.rglob("prediction.json"))
        gold = [GOLD / p.parent.relative_to(leaf) / "checks" / entry / "expected_output.json"
                for p in predictions]
        newest = max((p.stat().st_mtime for p in predictions + gold if p.is_file()), default=0.0)
        if newest > analysis.stat().st_mtime:
            stale.append(leaf)

if stale:
    raise RuntimeError(
        "predictions or gold are newer than their analysis in "
        f"{len(stale)} leaf/leaves; delete those analysis.json and re-run "
        "this cell:\n  " + "\n  ".join(str(p) for p in stale)
    )
print("all leaves scored, none stale")

## Endpoint 0 — dispatch, reach, order

Read before any score, from each session's `skill_trace.json`:

- **dispatch** — all three check skills invoked, and per check whether it was;
- **reach** — how many times `classify-panels` and `identify-panels` were invoked;
- **order** — the order in which the checks were first invoked;
- **spontaneous** — any skill outside the condition's closure;
- **`panel_classes`** (PM) — rows answered with an empty list: the field is
  unscored, so this is the only check that it is filled.

Reported, not a gate.

In [ ]:
def read_trace(path: Path) -> list[str]:
    try:
        entries = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return []
    return [e.get("skill") for e in entries if isinstance(e, dict) and e.get("skill")]


def read_answer(path: Path) -> dict | None:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


rows = []
for shape, entry, root in run_roots():
    if not root.is_dir():
        continue
    for arm, replicate, leaf in iter_leaves(root):
        arrangement = LABEL_TO_ARRANGEMENT[arm]
        expected = {entry, *CHECKS, *exp04.EXPECTED_SHARED[arrangement]}
        for trace in leaf.rglob("intermediates/skill_trace.json"):
            example_dir = trace.parent.parent
            skills = read_trace(trace)
            called = [s for s in skills if s in CHECKS]
            row = {
                "shape": shape,
                "arrangement": arrangement,
                "condition": condition(shape, arrangement),
                "replicate": replicate,
                "example": str(example_dir.relative_to(leaf)),
                "all_dispatched": set(CHECKS) <= set(called),
                **{f"dispatched {c}": c in called for c in CHECKS},
                "classify_calls": skills.count(CLASSIFY),
                "identify_calls": skills.count(IDENTIFY),
                "order": " > ".join(dict.fromkeys(called)),
                "spontaneous": tuple(sorted(set(skills) - expected)),
            }
            if shape == "pm":
                answer = read_answer(example_dir / "prediction.json") or {}
                outputs = answer.get("outputs") or []
                row["empty_panel_classes"] = sum(not r.get("panel_classes") for r in outputs)
                row["pm_rows"] = len(outputs)
            rows.append(row)

sessions = pd.DataFrame(rows)
(
    sessions.groupby("condition")[["all_dispatched", *(f"dispatched {c}" for c in CHECKS)]]
    .mean()
    .join(sessions.groupby("condition").size().rename("sessions"))
    .reindex(CONDITION_ORDER)
)

In [ ]:
# Reach: calls per session. 1 is one classification (or identification) shared
# by three checks; 3 would be one per check.
for skill in ("classify_calls", "identify_calls"):
    print(skill)
    print(sessions.groupby(["condition", skill]).size().unstack(fill_value=0).reindex(CONDITION_ORDER))
    print()

In [ ]:
# Order: if one dominates, a check's position and its identity are confounded.
sessions.groupby(["condition", "order"]).size().rename("sessions").reset_index()

In [ ]:
# Spontaneous: skills outside the condition's closure. Empty means none.
(
    sessions.explode("spontaneous").dropna(subset=["spontaneous"])
    .groupby(["condition", "spontaneous"]).size().rename("sessions").reset_index()
)

In [ ]:
# PM: rows whose panel_classes is empty.
pm_sessions = sessions[sessions["shape"] == "pm"]
pm_sessions.groupby("condition")[["empty_panel_classes", "pm_rows"]].sum().reindex(
    [c for c in CONDITION_ORDER if c.startswith("PM")])

## One row per property, per replicate, per example

Both shapes loaded and rewritten into per-check terms, then concatenated. From
here on a row is a `(condition, check, …)`.

In [ ]:
frames, layer_s, layer_1, layer_2, non_response = [], [], [], [], []
for shape, entry, root in run_roots():
    if not root.is_dir():
        continue
    runs = load_run_root(root, checklist=CHECKLIST, check=entry)
    frames.append(to_per_check(scores_frame(runs), shape))
    layer_s.append(to_per_check(layer_s_counts_by_example(runs), shape))
    layer_1.append(to_per_check(layer1_counts_by_example(runs), shape))
    layer_2.append(to_per_check(layer2_counts(runs), shape))
    non_response.append(to_per_check(non_response_counts(runs), shape))

scores = pd.concat(frames, ignore_index=True)
layer_s = pd.concat(layer_s, ignore_index=True)
layer_1 = pd.concat(layer_1, ignore_index=True)
layer_2 = pd.concat(layer_2, ignore_index=True)
non_response = pd.concat(non_response, ignore_index=True)

# The two shapes must be read on the same instance sets: per check, the same
# properties under CM and PM.
props = scores.groupby(["check", scores["condition"].str[:2]])["property"].agg(lambda s: frozenset(s))
for check in CHECKS:
    if ("CM" in props[check].index and "PM" in props[check].index
            and props[check]["CM"] != props[check]["PM"]):
        raise ValueError(f"{check}: CM and PM score different properties: "
                         f"{sorted(props[check]['CM'] ^ props[check]['PM'])}")

print(f"{len(scores)} score rows, {scores['check'].nunique()} checks, "
      f"{scores['replicate'].nunique()} replicate(s), {scores['condition'].nunique()} conditions")
scores.head()

## Non-response and failures, before any mean

An empty answer scores as a fully missing row set for every check — and under
PM an empty `outputs` loses all three at once, as a failed session does in both
shapes. A failed session wrote no prediction and is counted per session.

In [ ]:
(
    non_response.groupby(["check", "condition"])["empty"].sum()
    .unstack(fill_value=0).reindex(columns=CONDITION_ORDER)
)

In [ ]:
def benchmark(entry: str) -> list[str]:
    path = Path("../../soda_mmqc/data/checklist") / CHECKLIST / entry / "benchmark.json"
    return json.loads(path.read_text(encoding="utf-8"))["examples"]


missing = []
for shape, entry, root in run_roots():
    if not root.is_dir():
        continue
    wanted = benchmark(entry)
    for arm, replicate, leaf in iter_leaves(root):
        missing.append({
            "condition": condition(shape, LABEL_TO_ARRANGEMENT[arm]),
            "replicate": replicate,
            "sessions_without_prediction": sum(not (leaf / ex / "prediction.json").is_file()
                                               for ex in wanted),
            "sessions": len(wanted),
        })
missing = pd.DataFrame(missing)
missing.groupby("condition")[["sessions_without_prediction", "sessions"]].sum().reindex(CONDITION_ORDER)

## The outcomes, condition beside condition

Counts summed over examples and replicates. Read against the gate tables, not
against bar heights.

In [ ]:
plot_stacked_counts(
    layer_s, group="check", category="outcome", series="condition",
    order=LAYER_S_ORDER, series_order=CONDITION_ORDER, ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_1, group="check", category="layer1", series="condition",
    order=LAYER1_ORDER, series_order=CONDITION_ORDER, ylabel="instances",
    title="Layer 1 applicability calls, summed over properties, examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2["layer2"].isin(LAYER2_BINARY_ORDER)], group="check", category="layer2",
    series="condition", order=LAYER2_BINARY_ORDER, series_order=CONDITION_ORDER,
    ylabel="instances",
    title="Layer 2, binary properties, summed over properties, examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2["layer2"].isin(LAYER2_GRADED_ORDER)], group="check", category="layer2",
    series="condition", order=LAYER2_GRADED_ORDER, series_order=CONDITION_ORDER,
    ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over properties, examples and replicates",
)

## Hypothesis 1 — spurious panels per figure

A spurious panel is a label the answer states — in any check's list, for CM —
that the gold does not have, **counted once per figure** however many lists
repeat it (`experiments/exp-04-panel-major/panels.py`). Missing panels (gold
labels no list carries) and, for CM, partly missing panels are reported beside
it, without a direction.

Per figure, the count is averaged over the replicates that wrote a prediction;
PM − CM is paired by figure and bootstrapped over figures. **Fewer** if the 95%
interval lies below 0, **more** if above, **inconclusive** otherwise.

In [ ]:
rows = []
for shape, entry, root in run_roots():
    if not root.is_dir():
        continue
    for arm, replicate, leaf in iter_leaves(root):
        for prediction in leaf.rglob("prediction.json"):
            example = str(prediction.parent.relative_to(leaf))
            gold = json.loads((GOLD / example / "checks" / ENTRIES["pm"] / "expected_output.json")
                              .read_text(encoding="utf-8"))
            labels = [r["panel_label"] for r in gold["outputs"]]
            rows.append({
                "shape": shape,
                "arrangement": LABEL_TO_ARRANGEMENT[arm],
                "condition": condition(shape, LABEL_TO_ARRANGEMENT[arm]),
                "replicate": replicate,
                "example": example,
                **panels.panel_counts(labels, read_answer(prediction), shape),
            })
panel_rows = pd.DataFrame(rows)

totals = panel_rows.groupby("condition")[
    ["spurious_panels", "missing_panels", "partly_missing_panels", "gold_panels"]].sum()
totals["sessions"] = panel_rows.groupby("condition").size()
totals["figures_with_spurious"] = (
    panel_rows[panel_rows["spurious_panels"] > 0].groupby("condition")["example"].nunique())
totals.fillna(0).reindex(CONDITION_ORDER)

In [ ]:
def paired_bootstrap(diff: np.ndarray, rng) -> tuple[float, float, float]:
    boots = rng.choice(diff, size=(RESAMPLES, len(diff)), replace=True).mean(axis=1)
    return diff.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def direction(low, high) -> str:
    if pd.isna(low) or pd.isna(high):
        return "no data"
    return "fewer" if high < 0 else "more" if low > 0 else "inconclusive"


rng = np.random.default_rng(SEED)
per_figure = panel_rows.groupby(["arrangement", "shape", "example"])[
    ["spurious_panels", "missing_panels"]].mean()
rows = []
for arrangement in ARRANGEMENTS:
    for measure in ("spurious_panels", "missing_panels"):
        try:
            wide = per_figure[measure].loc[arrangement].unstack("shape")
        except KeyError:
            continue
        if not {"cm", "pm"} <= set(wide.columns):
            continue
        paired = wide.dropna()
        diff = (paired["pm"] - paired["cm"]).to_numpy()
        mean, low, high = paired_bootstrap(diff, rng) if len(diff) else (np.nan,) * 3
        rows.append({
            "arrangement": arrangement, "measure": measure,
            "cm_per_figure": paired["cm"].mean(), "pm_per_figure": paired["pm"].mean(),
            "difference": mean, "ci_low": low, "ci_high": high,
            "n_figures": len(diff), "unpaired": len(wide) - len(paired),
            "verdict": direction(low, high) if measure == "spurious_panels" else "reported",
        })
h1 = pd.DataFrame(rows)
h1

## The verdict rule for the gates

Within each arrangement, PM against CM, paired by example and bootstrapped over
the 38 examples.

- **Non-inferior** when the lower bound of the 95% CI lies above **−δ**
- **Degraded** when the interval lies entirely below −δ
- **Inconclusive** when it straddles −δ
- **Uninformative** — only for the cells pre-declared so — when the interval is
  wider than δ on either side (half-width > δ); then it is reported, not counted

Fixed sequence within each comparison, S → 1 → 2.

In [ ]:
def verdict(low, high, delta):
    if pd.isna(low) or pd.isna(high):
        return "no data"
    if low > -delta:
        return "non-inferior"
    if high < -delta:
        return "degraded"
    return "inconclusive"


def arrangement_of(cond: str) -> str:
    return cond.split(" ", 1)[1]


def mark_uninformative(frame: pd.DataFrame, endpoint: str, delta: float) -> pd.DataFrame:
    """Pre-declared cells whose interval cannot resolve δ are uninformative."""
    frame = frame.copy()
    declared = [
        any((endpoint, check, prop, arr) in UNINFORMATIVE for prop in (None, row_prop))
        for check, row_prop, arr in zip(
            frame["check"],
            frame["property"] if "property" in frame else [None] * len(frame),
            frame["arrangement"])
    ]
    frame["pre_declared"] = declared
    wide = frame["half_width"] > delta
    frame.loc[frame["pre_declared"] & wide, "verdict"] = "uninformative"
    return frame


def rate_gate(frame, *, category, numerator, denominator, group, delta, label):
    """One rate endpoint: PM against CM, within each arrangement."""
    out = []
    for check in CHECKS:
        sub = frame[frame["check"] == check]
        for variant, baseline in PAIRS.items():
            pair = sub[sub["condition"].isin((variant, baseline))]
            if pair["condition"].nunique() < 2:
                continue
            contrast = rate_contrast(
                pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
                baseline=baseline, variant=variant, category=category,
                numerator=numerator, denominator=denominator, group=group,
                resamples=RESAMPLES, seed=SEED,
            )
            contrast.insert(0, "arrangement", arrangement_of(variant))
            contrast.insert(0, "endpoint", label)
            out.append(contrast)
    if not out:
        return pd.DataFrame()
    result = pd.concat(out, ignore_index=True)
    result["half_width"] = (result["ci_high"] - result["ci_low"]) / 2
    result["verdict"] = [verdict(lo, hi, delta) for lo, hi in zip(result["ci_low"], result["ci_high"])]
    return mark_uninformative(result, label, delta)


GATE_COLUMNS = ["arrangement", "check", "difference", "ci_low", "ci_high",
                "half_width", "n_examples", "paired_fraction", "pre_declared", "verdict"]

## Gate 1 — layer S: the share of gold rows found

`correct_row / (correct_row + missing_row)` per check: the gate statistic of
exp-02 and exp-03. It leaves spurious rows out — they are hypothesis 1's — and
measures missing rows alone.

In [ ]:
gate1 = rate_gate(
    layer_s.drop(columns=["arrangement"]), category="outcome",
    numerator=("correct_row",), denominator=("correct_row", "missing_row"),
    group=("check", "list_key"), delta=DELTA["layer S"], label="layer S",
)
gate1[GATE_COLUMNS]

### Layer S, reported with spurious rows

`correct / (correct + missing + spurious)`, per check and condition: the one
rate that charges both errors. Mean of per-example rates (replicates pooled
first), percentile bootstrap over examples. **Reported, not a gate.**

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None):
    # Rate per (condition, check, example), replicates pooled first.
    counts = (frame.groupby(["condition", "check", "example", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


rng = np.random.default_rng(SEED)
rows = []
for label, denominator in (("found / (correct + missing)", ("correct_row", "missing_row")),
                           ("correct / (correct + missing + spurious)",
                            ("correct_row", "missing_row", "spurious_row"))):
    rates = per_example_rate(layer_s, "outcome", ("correct_row",), denominator)
    for (cond, check), sub in rates.groupby(["condition", "check"]):
        r = sub["rate"].dropna().to_numpy()
        boots = rng.choice(r, size=(RESAMPLES, len(r)), replace=True).mean(axis=1)
        rows.append({"rate": label, "condition": cond, "check": check, "mean": r.mean(),
                     "ci_low": np.percentile(boots, 2.5), "ci_high": np.percentile(boots, 97.5),
                     "n_examples": len(r)})
layer_s_rates = pd.DataFrame(rows)
layer_s_rates.pivot_table(index=["check", "condition"], columns="rate", values="mean").reindex(
    CONDITION_ORDER, level=1)

## Gate 2 — layer 1: applicability

`correct_applicable + correct_NA` over all profiled instances, per check.

In [ ]:
gate2 = rate_gate(
    layer_1.drop(columns=["arrangement"]), category="layer1",
    numerator=("correct_applicable", "correct_NA"), denominator=None,
    group=("check",), delta=DELTA["layer 1"], label="layer 1",
)
gate2[GATE_COLUMNS]

## Gate 3 — layer 2: the check-specific judgement

`mean_score` per property, on applicable instances, paired by example with
replicates averaged first. Per property, never pooled. The interval is
`difference ± 1.96·se`, as in exp-02 and exp-03.

In [ ]:
rows = []
for check in CHECKS:
    sub = scores[scores["check"] == check]
    for variant, baseline in PAIRS.items():
        pair = sub[sub["condition"].isin((variant, baseline))]
        if pair["condition"].nunique() < 2:
            continue
        contrast = arm_contrast(
            pair.drop(columns=["arm", "arrangement"]).rename(columns={"condition": "arm"}),
            baseline=baseline, variant=variant,
        )
        contrast.insert(0, "arrangement", arrangement_of(variant))
        rows.append(contrast)

gate3 = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if not gate3.empty:
    gate3["half_width"] = 1.96 * gate3["se"].astype(float)
    gate3["ci_low"] = gate3["difference"].astype(float) - gate3["half_width"]
    gate3["ci_high"] = gate3["difference"].astype(float) + gate3["half_width"]
    gate3["verdict"] = [verdict(lo, hi, DELTA["layer 2"])
                        for lo, hi in zip(gate3["ci_low"], gate3["ci_high"])]
    gate3 = mark_uninformative(gate3, "layer 2", DELTA["layer 2"])
gate3.sort_values("difference")[[
    "arrangement", "check", "property", "difference", "ci_low", "ci_high", "half_width",
    "n_examples", "paired_fraction", "pre_declared", "verdict",
]] if not gate3.empty else gate3

## Through the fixed sequence

Per arrangement and check, where the gate sequence stops: S, then 1, then 2.
Layer 2 is non-inferior when every property is (uninformative ones aside), and
degraded when any property is. Gates past a stop are description, not claims.

In [ ]:
def gate_state(verdicts) -> str:
    v = set(verdicts) - {"uninformative"}
    if not v:
        return "uninformative"
    if "degraded" in v:
        return "degraded"
    if v <= {"non-inferior"}:
        return "non-inferior"
    return "inconclusive"


rows = []
for arrangement in ARRANGEMENTS:
    for check in CHECKS:
        states = {}
        for label, gate in (("layer S", gate1), ("layer 1", gate2), ("layer 2", gate3)):
            sub = gate[(gate["arrangement"] == arrangement) & (gate["check"] == check)] if not gate.empty else gate
            states[label] = gate_state(sub["verdict"]) if len(sub) else "no data"
        stop = next((g for g, s in states.items() if s != "non-inferior"), None)
        rows.append({"arrangement": arrangement, "check": check, **states,
                     "stops_at": stop or "non-inferior at all three"})
sequence = pd.DataFrame(rows)
sequence

## The inputs to the choice rule

The preregistration's rule: PM becomes the series' contract if, in at least two
of the three arrangements, it states **fewer** spurious panels and **no check
degrades at any gate**; an arrangement with more spurious panels, or a
degradation, counts against PM; an uninformative cell counts neither way. This
table lays out the inputs per arrangement. The reading is the note's.

In [ ]:
rows = []
for arrangement in ARRANGEMENTS:
    seq = sequence[sequence["arrangement"] == arrangement]
    cells_in = lambda state: [f"{r['check']} {g}" for _, r in seq.iterrows()
                              for g in ("layer S", "layer 1", "layer 2") if r[g] == state]
    spur = h1[(h1["arrangement"] == arrangement) & (h1["measure"] == "spurious_panels")]
    rows.append({
        "arrangement": arrangement,
        "spurious panels": spur["verdict"].iloc[0] if len(spur) else "no data",
        "degraded cells": ", ".join(cells_in("degraded")) or "none",
        "inconclusive cells": ", ".join(cells_in("inconclusive")) or "none",
        "uninformative cells": ", ".join(cells_in("uninformative")) or "none",
    })
pd.DataFrame(rows)

## Secondary — the arrangements, within each shape

`A|B <- C_i <- D` and `A <- B <- C_i <- D` each against `A|B|C_i <- D`, per check,
layers S and 1: the baseline for the orchestration experiments. Paired by
example, bootstrapped. **Reported, not a gate**, so no verdict.

In [ ]:
rows = []
for shape in SHAPES:
    base = condition(shape, ARRANGEMENTS[0])
    for other in ARRANGEMENTS[1:]:
        variant = condition(shape, other)
        for label, frame, category, numerator, denominator, group in (
            ("layer S", layer_s, "outcome", ("correct_row",), ("correct_row", "missing_row"),
             ("check", "list_key")),
            ("layer 1", layer_1, "layer1", ("correct_applicable", "correct_NA"), None, ("check",)),
        ):
            for check in CHECKS:
                pair = frame[(frame["check"] == check) & frame["condition"].isin((base, variant))]
                if pair["condition"].nunique() < 2:
                    continue
                c = rate_contrast(
                    pair.drop(columns=["arm", "arrangement"]).rename(columns={"condition": "arm"}),
                    baseline=base, variant=variant, category=category, numerator=numerator,
                    denominator=denominator, group=group, resamples=RESAMPLES, seed=SEED)
                c.insert(0, "contrast", f"{variant} vs {ARRANGEMENTS[0]}")
                c.insert(0, "endpoint", label)
                rows.append(c)
arrangement_contrasts = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
arrangement_contrasts[["endpoint", "contrast", "check", "difference", "ci_low", "ci_high",
                       "n_examples"]] if rows else arrangement_contrasts

## Cost and time per figure

From each session's `tool_audit.json`. One session is one figure in both
shapes, so PM ÷ CM is a ratio of two sessions on the same figure, replicates
averaged per `(condition, example)` first, paired by example. Mean of
per-figure ratios, percentile bootstrap. **No direction predicted.**

In [ ]:
def read_usage(path: Path) -> dict:
    audit = json.loads(path.read_text(encoding="utf-8"))
    usage = audit.get("usage") or {}
    tokens = usage.get("usage") or {}
    return {
        "cost": usage.get("total_cost_usd"),
        "turns": usage.get("num_turns"),
        "seconds": (usage.get("duration_ms") or 0) / 1000,
        "input_tokens": sum(tokens.get(k) or 0 for k in
                            ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")),
        "cache_read_tokens": tokens.get("cache_read_input_tokens") or 0,
        "output_tokens": tokens.get("output_tokens") or 0,
    }


rows = []
for shape, entry, root in run_roots():
    if not root.is_dir():
        continue
    for arm, replicate, leaf in iter_leaves(root):
        for audit in leaf.rglob("intermediates/tool_audit.json"):
            rows.append({"shape": shape, "arrangement": LABEL_TO_ARRANGEMENT[arm],
                         "replicate": replicate,
                         "example": str(audit.parent.parent.relative_to(leaf)),
                         **read_usage(audit)})
usage = pd.DataFrame(rows)
QUANTITIES = ("cost", "input_tokens", "cache_read_tokens", "output_tokens", "turns", "seconds")
per_condition = usage.groupby(["shape", "arrangement", "example"])[list(QUANTITIES)].mean()
per_condition.groupby(level=[1, 0]).mean()

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for arrangement in ARRANGEMENTS:
    try:
        cm = per_condition.loc[("cm", arrangement)]
        pm = per_condition.loc[("pm", arrangement)]
    except KeyError:
        continue
    both = cm.join(pm, lsuffix="_cm", rsuffix="_pm", how="inner")
    for q in QUANTITIES:
        r = (both[f"{q}_pm"] / both[f"{q}_cm"]).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        if not len(r):
            continue
        boots = rng.choice(r, size=(RESAMPLES, len(r)), replace=True).mean(axis=1)
        rows.append({"arrangement": arrangement, "quantity": q, "pm_over_cm": r.mean(),
                     "ci_low": np.percentile(boots, 2.5), "ci_high": np.percentile(boots, 97.5),
                     "n_figures": len(r)})
pd.DataFrame(rows)

## Replicate variance, for sizing later margins

Per condition and check: the between-replicate variance of each example's
layer-S and layer-1 rate, averaged over examples, and the half-width a later
comparison of two such conditions at five replicates would plan for:
`1.96 · sqrt(mean_e(2σ²_e / 5) / n_e)` — the formula exp-03's P2 used.

In [ ]:
def replicate_variance(frame, category, numerator, denominator=None):
    counts = (frame.groupby(["condition", "check", "example", "replicate", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    rate = (num / den.where(den > 0)).rename("rate").reset_index()
    var = rate.groupby(["condition", "check", "example"])["rate"].var(ddof=1).rename("var").reset_index()
    out = var.groupby(["condition", "check"]).agg(mean_var=("var", "mean"), n_examples=("var", "count"))
    out["planned_half_width_5v5"] = 1.96 * np.sqrt(2 * out["mean_var"] / 5 / out["n_examples"])
    return out


pd.concat({
    "layer S": replicate_variance(layer_s, "outcome", ("correct_row",), ("correct_row", "missing_row")),
    "layer 1": replicate_variance(layer_1, "layer1", ("correct_applicable", "correct_NA")),
}, names=["endpoint"])

## What this does not do

It does not decide which contract the series keeps, and it computes no reading
of dispatch against the gates — that interpretation is fixed in
[`thinking/experiments/exp-04-panel-major.md`](../../thinking/experiments/exp-04-panel-major.md)
before the run, where it can be read and disagreed with.